# All + Time

In [ ]:
import os
import glob
import re
import matplotlib.pyplot as plt
from datetime import datetime
import pandas as pd  # Import pandas for DataFrame creation

def extract_and_plot_logs_multiple_methods(folder_paths, methods, keyword="resnet32"):
    # Create the directory where plots will be saved if it doesn't exist
    output_dir = f'./images/{keyword}/'
    os.makedirs(output_dir, exist_ok=True)

    # Initialize dictionaries to store combined data for each method
    combined_data = {method: {"X": [], "Accuracy_CNN": [], "Accuracy_NME": [], "Forgetting_CNN": [], "Time_Spent": []} for method in methods}

    # Iterate over each folder and corrsesponding method
    for folder_path, method in zip(folder_paths, methods):
        # Use glob to search for log files containing 'resnet32' in their name
        log_files = glob.glob(os.path.join(folder_path, f"*{keyword}*.log"))

        # Dictionary to store log file paths with X values
        log_files_with_x = {}

        # Extract X values from filenames and store them in a dictionary
        for log_file in log_files:
            # Use regex to extract the X value (e.g., memory_1993_resnet32.log)
            match = re.search(r'memory_(\d+)_1993_{a}'.format(a=keyword), log_file)
            if match:
                x_value = int(match.group(1))  # Extract X as an integer
                log_files_with_x[log_file] = x_value

        # Sort the log files by X value
        sorted_log_files = sorted(log_files_with_x.keys(), key=lambda x: log_files_with_x[x])

        # Loop through each sorted log file and extract the relevant values
        for log_file in sorted_log_files:
            with open(log_file, 'r') as f:
                lines = f.readlines()

                # Extract the first and last timestamp for the time spent
                first_line = lines[0]
                last_line = lines[-1]
                first_timestamp = re.search(r"(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2},\d{3})", first_line).group(1)
                last_timestamp = re.search(r"(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2},\d{3})", last_line).group(1)
                # Convert timestamps to datetime objects
                time_format = "%Y-%m-%d %H:%M:%S,%f"
                start_time = datetime.strptime(first_timestamp, time_format)
                end_time = datetime.strptime(last_timestamp, time_format)
                time_spent = (end_time - start_time).total_seconds() / 60  # Time in minutes

                # Extract the metrics from the log
                avg_accuracy_cnn = None
                avg_accuracy_nme = None
                forgetting_cnn = None

                for line in lines:
                    if "Average Accuracy (CNN)" in line:
                        avg_accuracy_cnn = float(re.search(r": ([\d.]+)", line).group(1))
                    elif "Average Accuracy (NME)" in line:
                        avg_accuracy_nme = float(re.search(r": ([\d.]+)", line).group(1))
                    elif "Forgetting (CNN)" in line:
                        forgetting_cnn = float(re.search(r": ([\d.]+)", line).group(1))

                # Ensure we got all three metrics before adding them to the combined data
                if avg_accuracy_cnn is not None and avg_accuracy_nme is not None and forgetting_cnn is not None:
                    combined_data[method]["X"].append(log_files_with_x[log_file])
                    combined_data[method]["Accuracy_CNN"].append(avg_accuracy_cnn)
                    combined_data[method]["Accuracy_NME"].append(avg_accuracy_nme)
                    combined_data[method]["Forgetting_CNN"].append(forgetting_cnn)
                    combined_data[method]["Time_Spent"].append(time_spent)

    # Sort the combined data by X values for each method
    for method in methods:
        sorted_data = sorted(zip(combined_data[method]["X"],
                                 combined_data[method]["Accuracy_CNN"],
                                 combined_data[method]["Accuracy_NME"],
                                 combined_data[method]["Forgetting_CNN"],
                                 combined_data[method]["Time_Spent"]),
                             key=lambda x: x[0])

        combined_data[method]["X"] = [x[0] for x in sorted_data]
        combined_data[method]["Accuracy_CNN"] = [x[1] for x in sorted_data]
        combined_data[method]["Accuracy_NME"] = [x[2] for x in sorted_data]
        combined_data[method]["Forgetting_CNN"] = [x[3] for x in sorted_data]
        combined_data[method]["Time_Spent"] = [x[4] for x in sorted_data]

    # Define font sizes for consistency
    title_fontsize = 24
    label_fontsize = 20
    legend_fontsize = 12

    default_colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
    method_colors = {
        method: default_colors[i % len(default_colors)]
        for i, method in enumerate(methods)
    }
    # Now plot the combined data for all methods

    # Plot Average Accuracy (CNN)
    plt.figure(figsize=(8, 4))
    for method in methods:
        # Use a dashed line for 'der' and 'foster', solid for others
        linestyle = '--' if method in ['DER','FOSTER','MEMO'] else '-'
        plt.plot(combined_data[method]["X"], combined_data[method]["Accuracy_CNN"],
            label=method, marker='o', linestyle=linestyle, color=method_colors[method])
    plt.title("Average Accuracy (CNN)", fontsize=title_fontsize)
    plt.xlabel("Memory Size", fontsize=label_fontsize)
    plt.ylabel("Accuracy", fontsize=label_fontsize)
    plt.grid(True)
    plt.legend(loc='best', fontsize=legend_fontsize, ncol=2)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'average_accuracy_cnn_combined.pdf'))
    plt.savefig(os.path.join(output_dir, 'average_accuracy_cnn_combined.png'))
    plt.show()

    # Plot Average Accuracy (NME)
    plt.figure(figsize=(8, 4))
    for method in methods:
        linestyle = '--' if method in ['DER','FOSTER','MEMO'] else '-'
        plt.plot(combined_data[method]["X"], combined_data[method]["Accuracy_NME"],
                 label=method, marker='o', linestyle=linestyle,color=method_colors[method])
    plt.title("Average Accuracy (NME)", fontsize=title_fontsize)
    plt.xlabel("Memory Size", fontsize=label_fontsize)
    plt.ylabel("Accuracy", fontsize=label_fontsize)
    plt.grid(True)
    plt.legend(loc='best', fontsize=legend_fontsize, ncol=2)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'average_accuracy_nme_combined.pdf'))
    plt.savefig(os.path.join(output_dir, 'average_accuracy_nme_combined.png'))
    plt.show()

    # Plot Forgetting (CNN)
    plt.figure(figsize=(8, 4))
    for method in methods:
        linestyle = '--' if method in ['DER','FOSTER','MEMO'] else '-'
        plt.plot(combined_data[method]["X"], combined_data[method]["Forgetting_CNN"],
                 label=method, marker='o', linestyle=linestyle,color=method_colors[method])
    plt.title("Forgetting (CNN)", fontsize=title_fontsize)
    plt.xlabel("Memory Size", fontsize=label_fontsize)
    plt.ylabel("Forgetting (CNN)", fontsize=label_fontsize)
    plt.grid(True)
    plt.legend(loc='best', fontsize=legend_fontsize, ncol=2)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'forgetting_cnn_combined.pdf'))
    plt.savefig(os.path.join(output_dir, 'forgetting_cnn_combined.png'))
    plt.show()

    # Plot Time Spent
    plt.figure(figsize=(8, 4))
    for method in methods:
        # Special handling for 'bic'
        linestyle = '--' if method in ['DER','FOSTER','MEMO'] else '-'
        plt.plot(combined_data[method]["X"], combined_data[method]["Time_Spent"],
                 label=method, marker='*', linestyle=linestyle,color=method_colors[method])
    plt.title("Time Spent", fontsize=title_fontsize)
    plt.xlabel("Memory Size", fontsize=label_fontsize)
    plt.ylabel("Time Spent (min.)", fontsize=label_fontsize)
    plt.grid(True)
    plt.legend(loc='best', fontsize=legend_fontsize, ncol=2)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'time_spent_combined.pdf'))
    plt.savefig(os.path.join(output_dir, 'time_spent_combined.png'))
    plt.show()

    # Create a DataFrame comparing the Average Accuracy (CNN) across methods
    accuracy_cnn_df = pd.DataFrame()

    for method in methods:
        temp_df = pd.DataFrame({
            'Memory Size': combined_data[method]["X"],
            f'Accuracy_CNN_{method}': combined_data[method]["Accuracy_CNN"]
        })
        if accuracy_cnn_df.empty:
            accuracy_cnn_df = temp_df
        else:
            accuracy_cnn_df = pd.merge(accuracy_cnn_df, temp_df, on='Memory Size', how='outer')

    # Sort the DataFrame by Memory Size
    accuracy_cnn_df = accuracy_cnn_df.sort_values(by='Memory Size').reset_index(drop=True)
    accuracy_cnn_df.set_index('Memory Size', inplace=True)

    # Save the DataFrame to a CSV file
    accuracy_cnn_df.to_csv(os.path.join(output_dir, 'average_accuracy_cnn_comparison.csv'))

    return accuracy_cnn_df  # Return the DataFrame




In [ ]:
# List of folder paths, corresponding to each method
folder_paths = [
    "./logs/der/cifar100/0/10/",
    "./logs/foster/cifar100/0/10/",
    "./logs/memo/cifar100/0/10/",
    "./logs/replay/cifar100/0/10/",
    "./logs/icarl/cifar100/0/10/",
    "./logs/bic/cifar100/0/10/",
    "./logs/wa/cifar100/0/10/",
    "./logs/wsc/cifar100/0/10/",

]

# Corresponding methods for each folder path
methods = ["DER", "FOSTER", "MEMO", "Replay", "iCaRL", "BiC", "WA", "WSC"]

# Call the function to combine logs and plot results with method names
extract_and_plot_logs_multiple_methods(folder_paths, methods, keyword="resnet32")
